In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('sentiment_analysis_corpus.csv')

In [5]:
df.head()

,text,labels,preds,feedback,retrain_labels,retrained_preds
0,Wow... Loved this place.,2,NaN,NaN,NaN,NaN
1,Crust is not good.,0,NaN,NaN,NaN,NaN
2,Not tasty and the texture was just nasty.,0,NaN,NaN,NaN,NaN
3,Stopped by during the late May bank holiday of...,2,NaN,NaN,NaN,NaN
4,The selection on the menu was great and so wer...,2,NaN,NaN,NaN,NaN


In [6]:
df = df[['text', 'labels']]

In [7]:
df.head()

,text,labels
0,Wow... Loved this place.,2
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,2
4,The selection on the menu was great and so wer...,2


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100858 entries, 0 to 100857
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    100000 non-null  object
 1   labels  100858 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.5+ MB


In [9]:
df.isnull().sum()

text      858
labels      0
dtype: int64

In [10]:
df.dropna(inplace=True)

In [13]:
df.duplicated().sum()

np.int64(0)

In [12]:
df = df.drop_duplicates()

0: Negative  
1: Neutral  
2: Positive  

In [1]:
from nltk.corpus import stopwords
from numpy import array
from tensorflow.keras.preprocessing.text import one_hot, Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Dense
from tensorflow.keras.layers import Flatten, GlobalMaxPooling1D, Embedding, Conv1D, LSTM
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [14]:
stop_words = set(stopwords.words('english'))
stop_words.discard('not')


In [15]:
def preprocessed_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [words for words in tokens if words.isalpha() and words not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(words) for words in tokens]
    return " ".join(tokens)

In [16]:
df['text'].info()

<class 'pandas.core.series.Series'>
Index: 94983 entries, 0 to 100857
Series name: text
Non-Null Count  Dtype 
--------------  ----- 
94983 non-null  object
dtypes: object(1)
memory usage: 1.4+ MB


In [17]:
df['text'] = df['text'].apply(lambda x: preprocessed_text(x)) #30 min 20 sec time for processing

In [18]:
df.head()

,text,labels
0,wow loved place,2
1,crust not good,0
2,not tasty texture nasty,0
3,stopped late may bank holiday rick steve recom...,2
4,selection menu great price,2


In [19]:
x = df['text']
y = df['labels']

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=True)

In [21]:
y_train.value_counts(normalize = True)

labels
0    0.354776
2    0.331693
1    0.313531
Name: proportion, dtype: float64

In [22]:
word_tokanizer = Tokenizer()
word_tokanizer.fit_on_texts(x_train)
x_train = word_tokanizer.texts_to_sequences(x_train)
x_test = word_tokanizer.texts_to_sequences(x_test)

In [23]:
vocab_length = len(word_tokanizer.word_index) + 1

In [24]:
vocab_length

31700

In [25]:
maxlen = 100
x_train = pad_sequences(x_train, padding='post', maxlen=maxlen)
x_test = pad_sequences(x_test, padding='post', maxlen=maxlen)

In [27]:
from numpy import asarray
from numpy import zeros
embd_dict = dict()
glove_file = open('glove.6B.100d.txt', encoding="utf8")
for line in glove_file:
    records = line.split()
    word = records[0]
    vector_dimensions = asarray(records[1:], dtype='float32')
    embd_dict [word] = vector_dimensions
glove_file.close()

In [28]:
embedding_matrix = zeros((vocab_length, 100))
for word, index in word_tokanizer.word_index.items():
    embedding_vector = embd_dict.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

# Print Embedding Matrix shape
embedding_matrix.shape

(31700, 100)

In [29]:
import tensorflow as tf

In [38]:
import datetime

In [39]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [30]:

from keras.layers import LSTM

lstm_model = Sequential()

embedding_layer = Embedding(input_dim = vocab_length, output_dim = 100, weights=[embedding_matrix], 
                            input_length = 100, trainable = False, input_shape = (100, ))
lstm_model.add(embedding_layer)
lstm_model.add(LSTM(128, return_sequences=True))
lstm_model.add(LSTM(64, return_sequences=False))
lstm_model.add(Dense(3, activation = 'softmax'))

optimizer = tf.keras.optimizers.Adam(learning_rate = 0.005)
lstm_model.compile(optimizer = optimizer, loss = 'sparse_categorical_crossentropy', metrics = ['acc'])


print(lstm_model.summary())


d:\GUVI\visual_studio\Final_Project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
d:\GUVI\visual_studio\Final_Project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 100)       │     3,170,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 128)       │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,336,851 (12.73 MB)

 Trainable params: 166,851 (651.76 KB)

 Non-trainable params: 3,170,000 (12.09 MB)

None


In [37]:
from tensorflow.keras.utils import plot_model
plot_model(lstm_model, to_file='model_visualization.png', show_shapes=True, show_layer_names=True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [36]:
pip install pydot

Note: you may need to restart the kernel to use updated packages.


In [40]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
lstm_model_history = lstm_model.fit(x_train, y_train, batch_size=128, epochs=5, verbose=1, validation_split=0.2, callbacks=[tensorboard_callback])


Epoch 1/5
475/475 ━━━━━━━━━━━━━━━━━━━━ 103s 213ms/step - acc: 0.3924 - loss: 1.0713 - val_acc: 0.5312 - val_loss: 0.9629
Epoch 2/5
475/475 ━━━━━━━━━━━━━━━━━━━━ 97s 204ms/step - acc: 0.5313 - loss: 0.9409 - val_acc: 0.5926 - val_loss: 0.8384
Epoch 3/5
475/475 ━━━━━━━━━━━━━━━━━━━━ 97s 203ms/step - acc: 0.6465 - loss: 0.7740 - val_acc: 0.6836 - val_loss: 0.7174
Epoch 4/5
475/475 ━━━━━━━━━━━━━━━━━━━━ 98s 206ms/step - acc: 0.7278 - loss: 0.6330 - val_acc: 0.7254 - val_loss: 0.6362
Epoch 5/5
475/475 ━━━━━━━━━━━━━━━━━━━━ 97s 204ms/step - acc: 0.7940 - loss: 0.4996 - val_acc: 0.7521 - val_loss: 0.6057


In [88]:
# %tensorboard --logdir logs

In [93]:
#evaluating model performance
score = lstm_model.evaluate(x_test, y_test, verbose=1)

# Model Performance
print("Test Score:", score[0])
print("Test Accuracy:", score[1])

594/594 ━━━━━━━━━━━━━━━━━━━━ 22s 37ms/step - acc: 0.7795 - loss: 0.6392
Test Score: 0.6249448657035828
Test Accuracy: 0.7858608961105347


In [80]:
tf.__version__

'2.19.0'

In [82]:
import tensorboard
tensorboard.__version__

'2.19.0'

In [84]:
rmdir /s /q logs

'rm' is not recognized as an internal or external command,
operable program or batch file.


In [122]:
user_input = input()
user_input = preprocessed_text(user_input)
input_text = [user_input]
# text_out = word_tokanizer.fit_on_texts(input_text)
text_out = word_tokanizer.texts_to_sequences(input_text)
text_out_padded = pad_sequences(text_out, padding = 'post', maxlen = 100)
my_prediction = lstm_model.predict(text_out_padded)
print(input_text)
print('Prediction Mteric ',my_prediction)
print('Sentiment',my_prediction.argmax())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
['used product already opened']
Prediction Mteric  [[0.6592649  0.01288361 0.32785153]]
Sentiment 0


In [123]:
import pickle
with open('my_model1.pkl', 'wb') as model_file:
    pickle.dump(lstm_model, model_file)

with open('my_keras_tokens1.pkl', 'wb') as token_file:
    pickle.dump(word_tokanizer, token_file )